# Sandbox

A scratch notebook: somewhere to try a query without disturbing 01, 02 or 03. §4 is the reason it exists today — the playlist comparison Marc asked for — and §5 is deliberately empty and meant to stay that way in git.

It reads the warehouse and makes no Spotify API call. The profile comes from `SPOT_PROFILE`, so `make report-05 PROFILE=<slug>` chooses the person and this notebook names nobody.

## 1. Connect

`setup()` resolves the profile, opens the connection and configures the plotting theme. It is the only cell that knows how to reach the database.

In [ ]:
import pandas as pd
from IPython.display import Markdown

from spotify_lakehouse import playlists
from spotify_lakehouse.notebook import setup

ctx = setup()
display(
    Markdown(
        f"Profile **`{ctx.profile}`** · session **`{ctx.session}`** · "
        f"schemas `{ctx.stg_schema}` and `{ctx.mart_schema}`."
    )
)

**A query Marc added.** Playlist name, track total, and the public/collaborative flags straight from `dim_playlist`. Kept as written; §0 wants a markdown cell before every code cell, so this is its caption rather than an edit to the query.

In [ ]:
ctx.frame("""
select p.playlist_name, p.track_total, p.is_public, p.is_collaborative
from {mart}.dim_playlist as p
where p.is_current and p.playlist_key <> -1
  and not exists (select 1 from {mart}.fct_playlist_membership as m
                  where m.playlist_key = p.playlist_key)
order by p.track_total desc
""")

## 2. The API, in one screen

Four ways to ask the warehouse a question. **It is `ctx.conn`**, not `ctx.con`.

| Call | Returns | One-liner |
|---|---|---|
| `ctx.frame(sql, params)` | `pandas.DataFrame` | `ctx.frame("select * from {mart}.dim_playlist limit 5")` |
| `ctx.rows(sql, params)` | list of tuples | `ctx.rows("select count(*) from {mart}.fct_play_event")` |
| `ctx.scalar(sql, params)` | one value | `ctx.scalar("select count(*) from {mart}.dim_content")` |
| `ctx.conn` | the psycopg connection | `with ctx.conn.cursor() as cur: ...` |

`{stg}` and `{mart}` are replaced with this session's schema names, so a query written here runs unchanged in a worktree. Double any literal brace.

In [ ]:
example = ctx.frame(
    "select playlist_name, track_total from {mart}.dim_playlist "
    "where is_current and playlist_key > 0 order by track_total desc limit 5"
)
display(example)
display(
    Markdown(
        f"`dim_content` holds **{ctx.scalar('select count(*) from {mart}.dim_content'):,}** rows; "
        f"`ctx.conn` is a `{type(ctx.conn).__name__}`."
    )
)

## 3. What you can query

Every table, its row count and a runnable example live in **notebook 01 §11, The Queryable Catalog** — introspected at run time, so it cannot go stale. This section deliberately does not copy it: a second list would drift the first time a model landed.

`make report-01` renders it.

## 4. Playlist comparison

The question: **which songs are in one playlist and not in the other.** `miss` is directional (data-contracts §5), so every answer below says which direction it is measuring.

### 4.1 The playlists that were loaded

There is still no *owner* column — a playlist's owner is a person who is not necessarily Marc, this
repository is public, and owner identity is discarded before anything is stored (R-054). What
answers "is this mine?" instead is **`is_owned_by_profile`**, a boolean `scrub` computes from
`owner.id` *before* discarding it (R-058). No identifier is written; only the bit survives.

<!-- caption: Playlists by ownership against whether their contents could be loaded -->

In [ ]:
catalog = playlists.load_playlists(ctx)
display(catalog.head(25))
owned = int(catalog["is_owned_by_profile"].fillna(False).sum())
with_rows = int((catalog["tracks_loaded"] > 0).sum())
display(
    Markdown(
        f"**{len(catalog)}** playlists, **{int(catalog['tracks_loaded'].sum()):,}** tracks loaded. "
        f"**{owned}** are this profile's own; **{with_rows}** have contents in the warehouse."
    )
)
display(Markdown("**Ownership against whether the contents could be loaded:**"))
display(
    catalog.assign(loaded=catalog["tracks_loaded"] > 0)
    .groupby(["is_owned_by_profile", "loaded"])
    .size()
    .rename("playlists")
    .reset_index()
)

### 4.2 Resolving the two names

Resolved case-insensitively, and **never guessed**: a name matching zero playlists or more than one prints the candidates and stops, because silently picking one answers a different question than the one asked.

In [ ]:
# Marc's own words. Matching folds apostrophe variants, because Spotify stores the name with a
# typographic apostrophe and an ASCII one finds nothing (R-054 F7).
#
# 🚨 The right-hand side is the COPY Marc made in the Spotify app, not the original. The original
# `Connor\u2019s Playlist` is one of the 78 playlists Spotify refuses (403), so it has 0 tracks
# loaded and cannot be compared. Three playlists now contain "Connor", which is why the exact-name
# rule is load-bearing rather than a nicety (R-054 F3).
LEFT_NAME, RIGHT_NAME = "Smith", "Connor's Playlist (MFA)"
membership = playlists.load_membership(ctx)

# Resolve against EVERY playlist, not only those with rows loaded. A playlist Spotify refused has no
# membership, so resolving against the membership frame makes it invisible — and a leftover
# incidental match then looks unique and answers a different question (R-054).
matches, problems = {}, []
for needle in (LEFT_NAME, RIGHT_NAME):
    try:
        matches[needle] = playlists.resolve(catalog, needle)
    except playlists.AmbiguousPlaylist as exc:
        problems.append(str(exc))

for needle, match in matches.items():
    loaded = int(catalog.loc[catalog["playlist_name"] == match.name, "tracks_loaded"].sum())
    how = "exact name" if match.exact else "substring match"
    display(Markdown(f"**{needle}** → **{match.name}** ({how}, {loaded:,} tracks loaded)"))
    if match.warning:
        display(Markdown(f"> [!WARNING]\n> {match.warning}"))
    if loaded == 0:
        problems.append(
            f"{match.name!r} has no tracks loaded, so it cannot be compared. Spotify refused it "
            "(403) — see the load report."
        )

for problem in problems:
    display(Markdown(f"> [!WARNING]\n> {problem}"))

shown = sorted(
    set(playlists.candidates(catalog, LEFT_NAME))
    | set(playlists.candidates(catalog, RIGHT_NAME))
    | set(playlists.candidates(catalog, "Connor"))
)
display(Markdown("**Every playlist whose name contains either word:**"))
display(
    catalog[catalog["playlist_name"].isin(shown)][["playlist_name", "track_total", "tracks_loaded"]]
)
ready = len(matches) == 2 and not problems

### 4.3 The answer

<!-- caption: Tracks present in the first playlist and absent from the second, by artist and title -->

In [ ]:
if ready:
    left_name, right_name = matches[LEFT_NAME].name, matches[RIGHT_NAME].name
    left = membership[membership["playlist_name"] == left_name]
    right = membership[membership["playlist_name"] == right_name]
    answer = playlists.misses(
        left, right, left_name=left_name, right_name=right_name, tier="content_uri"
    )
    display(
        Markdown(
            f"### {answer.label}\n\n"
            f"**{answer.miss_count}** of **{answer.left_size}** tracks. "
            f"{answer.coverage_note}."
        )
    )
    display(answer.rows[["artist", "title"]])
else:
    display(
        Markdown(
            "> [!WARNING]\n> Both names must resolve to exactly one playlist "
            "before §4.3 can run. See §4.2."
        )
    )

**The same list, narrowed twice.** The table above is every track whose *URI* is absent from the
other playlist, and R-059 showed that over-counts: a track can be present under a different URI.
Narrowing it in order of how much the evidence is worth:

| list | a track leaves the list when |
|---|---|
| **no URI and no ISRC match** | the other side has the same recording under a different URI |
| **no match at any tier** | ...or the normalized name matches as well |

⚠️ **The second list is smaller, and it leans on the loose name tier.** R-060 measured that tier
over-matching on real data — a 1982 `Peer Gynt` recording and the "Can I Kick It" intro edit of
`Bonita Applebum` both collide with the standard versions. So the rows that sit *between* the two
lists are named below rather than quietly dropped, and §4.8 carries the full split.

<!-- caption: Tracks in the left playlist with neither identifier on the other side, name-only clears marked -->


In [ ]:
if ready:
    identifier = playlists.identifier_misses(
        left, right, left_name=left_name, right_name=right_name
    )
    strict = identifier.strict
    display(
        Markdown(
            f"### {identifier.label}\n\n"
            f"**{answer.miss_count}** miss by URI → **{identifier.miss_count}** also miss by ISRC "
            f"→ **{len(strict)}** miss at every tier."
        )
    )
    display(identifier.rows[["artist", "title", "album", "isrc", "cleared_by_name"]])
    if len(identifier.cleared):
        display(
            Markdown(
                f"**The {len(identifier.cleared)} between the two lists**, present on the other "
                "side only by a normalized-name match. That is the tier R-060 measured "
                "over-matching, so these are worth an eye rather than a silent drop:"
            )
        )
        display(identifier.cleared[["artist", "title", "album", "isrc"]])
    else:
        display(
            Markdown("The two lists are identical here: nothing is cleared by the name tier alone.")
        )
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

### 4.4 All three tiers, side by side

data-contracts §5: *the gap between tiers is itself the story* — tier 1 to tier 2 is re-issue churn, tier 2 to tier 3 is genuine ambiguity.

⚠️ **Tier 2's reach depends on which side you ask about.** Across `dim_content` as a whole, ISRC lands only on API-resolved tracks — a little over 1% of rows. But a *playlist* payload carries `external_ids.isrc` on the track object itself, so for playlist membership tier 2 covers almost everything. The `tier_coverage` column prints the share for the playlist in hand rather than assuming either figure, because the unqualified sentence "tier 2 only covers API-resolved tracks" is true of the warehouse and misleading about this comparison.

In [ ]:
if ready:
    display(playlists.tier_table(left, right, left_name=left_name, right_name=right_name))
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

### 4.5 The other direction

Marc asked for one direction. The reverse costs nothing and is the check that the first answer is not simply an artefact of one playlist being larger than the other.

In [ ]:
if ready:
    reverse = playlists.misses(
        right, left, left_name=right_name, right_name=left_name, tier="content_uri"
    )
    display(
        Markdown(
            f"### {reverse.label}\n\n**{reverse.miss_count}** of **{reverse.left_size}** tracks."
        )
    )
    display(reverse.rows[["artist", "title"]])
    display(playlists.tier_table(right, left, left_name=right_name, right_name=left_name))
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

### 4.6 How much of this the warehouse already knew

A playlist can contain tracks nobody has played, and `dim_content` is built from plays and API track lookups. This is the share of playlist tracks that already had a row.

In [ ]:
display(playlists.orphan_summary(ctx))
display(membership.groupby("in_dim_content").size().rename("tracks").reset_index())

### 4.7 Why the three tiers disagree

**As of R-060 the ladder nests for this pair — and that is a change, not a correction of R-059.**
R-059 measured the *loose* tier 3 missing more than the stricter tier 2 (31 against 30), which a
nested ladder cannot do, and `data-contracts.md` §5 records it. The cause was the `JAŸ-Z` diaeresis
surviving on one side of the comparison. R-060 made the tier-3 key fold diacritics and pick its
winning observation order-invariantly, so `Otis` and `Ni**as In Paris` now match at tier 3 and that
violation is gone.

What remains is the other mechanism, and it points the *other* way: **one recording carrying two
ISRCs** across two album releases, so tier 2 misses a track the name key catches. That makes tier 2
miss more than tier 3, which is not a nesting violation.

🚨 The relations are computed below rather than asserted, so this section reports the warehouse as
it stands rather than as it stood when the prose was written.


In [ ]:
tiers = {}
if ready:
    for tier in playlists.TIERS:
        result = playlists.misses(
            left, right, left_name=left_name, right_name=right_name, tier=tier
        )
        tiers[tier] = set(result.rows["content_uri"])
    display(
        pd.DataFrame(
            [
                {
                    "relation": "tier 2 (isrc) is a subset of tier 1 (uri)",
                    "holds": tiers["isrc"] <= tiers["content_uri"],
                },
                {
                    "relation": "tier 3 (name) is a subset of tier 1 (uri)",
                    "holds": tiers["content_match_key"] <= tiers["content_uri"],
                },
                {
                    "relation": "tier 3 (loose) is a subset of tier 2 (strict)",
                    "holds": tiers["content_match_key"] <= tiers["isrc"],
                },
            ]
        )
    )
    display(
        Markdown(
            f"Missed at tier 3 but matched at tier 2: "
            f"**{len(tiers['content_match_key'] - tiers['isrc'])}**. "
            f"The reverse: **{len(tiers['isrc'] - tiers['content_match_key'])}**."
        )
    )
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

### 4.8 The two lists

**List A — absent at every tier.** No URI, no ISRC, no normalized-name match on the other side.
This is the list to act on.

**List B — the same recording under a different URI.** Matched at tier 2 and/or tier 3 but not tier
1, with the album on *each* side so "sourced from a different album" is legible rather than asserted.

⚠️ The sizes below are measured, not assumed. §4.3 now shows the intermediate list — no URI and
no ISRC match — so the narrowing from tier-1 misses down to List A is visible in one place.

<!-- caption: Tracks in the left playlist absent from the right at every tier -->

In [ ]:
if ready:
    split = playlists.split_misses(left, right, left_name=left_name, right_name=right_name)
    display(
        Markdown(
            f"### {split.absent_label}\n\n"
            f"**{len(split.absent)}** tracks, of **{len(split.absent) + len(split.rehoused)}** "
            f"that miss at tier 1."
        )
    )
    display(split.absent[["artist", "title", "album", "isrc"]])
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

<!-- caption: Tracks present on both sides under different URIs, with the album on each -->

In [ ]:
if ready:
    display(
        Markdown(
            f"### {split.rehoused_label}\n\n"
            f"**{len(split.rehoused)}** tracks. A row here is a cataloguing difference, not an "
            f"absence."
        )
    )
    columns = ["artist", "title", "matched_tier", "album", "album_other_side", "isrc"]
    display(split.rehoused[[c for c in columns if c in split.rehoused.columns]])
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

### 4.9 The named validation case: `Heaven`

Marc's test case, and a clean one: **both sides credit `Los Lonely Boys`** — he initially recalled
Los Lobos and corrected it, and the stored payload agrees with the correction. Because the artist is
identical on both sides, a tier-3 non-match here could only come from the **title** string, so this
isolates the normalized-name tier with nothing else to hide behind.

The albums differ, which is exactly the case Marc asked to see separated out.

In [ ]:
if ready:
    both = membership[
        (membership["title"] == "Heaven")
        & (membership["playlist_name"].isin([left_name, right_name]))
    ]
    display(both[["playlist_name", "artist", "title", "album", "isrc", "content_match_key"]])
    tiers_hit = [
        tier
        for tier in playlists.TIERS
        if "Heaven"
        not in set(
            playlists.misses(
                left, right, left_name=left_name, right_name=right_name, tier=tier
            ).rows["title"]
        )
    ]
    display(
        Markdown(
            f"`Heaven` matches at: **{', '.join(tiers_hit) or 'no tier'}**. "
            f"The two sides agree on the artist, so tier 3 is testing the title alone — "
            f"and it matched, which is the normalization working."
        )
    )
else:
    display(Markdown("> [!WARNING]\n> Not run: see §4.2."))

## 5. Scratch

Empty on purpose. Add a cell, ask the warehouse something, and keep or discard it — a diff of this notebook should show only what you added.

The warehouse connection is closed at the end of the run.

In [ ]:
ctx.close()